# Q1 2026 ablation: Rejected New Writers vs HCP universe

Kerendia PD, RJ and RV NBRx per HCP and month for Jan to Mar 2026 (claims cohorts c19 to c21) on the NBA 3.1 HCP universe. Defines New Writers (PD NBRx > 0) and Rejected New Writers (RJ or RV NBRx > 0) and tests whether FTP and copay features separate Rejected New Writers from the rest of the universe.

* NBRx rule = `HCP_Target_Variable`: first claim per patient-day; new if the patient's first claim or the previous claim is more than 365 days earlier. Applied separately within each claim type (PD, RJ, RV).
* Features use each cohort's lookback (ends two months before the prediction month), so they are known before the labelled month.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import pandas as pd, numpy as np
import pyspark.sql.functions as F
from pyspark.sql.window import Window

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"
TARGET_PATH = f"{VOL31}/notebook_output/target_variable/target_variable"
MA_DIR = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/market_access_nba_3_1"
FTP_PATH = f"{MA_DIR}/kerendia_nba_3_1_hcp_model_ftp_features"
CPY_PATH = f"{MA_DIR}/kerendia_nba_3_1_hcp_model_copay_features"
PRIOR_WRITING_PATH = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features/kerendia_prior_writing"
MONTHLY_RX_PATH = f"{VOL31}/hcp_level_model/dev_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx"
OUTPUT_PATH = f"{VOL31}/notebook_output/target_variable/ablation_q1_2026/hcp_month_kerendia_nbrx_pd_rj_rv"

Q1_START, Q1_END = "2026-01-01", "2026-03-31"
DATA_START_NBRX = str(pd.Timestamp(Q1_START) - pd.Timedelta(days=365))[:10]   # 365-day NBRx lookback
CLAIM_TYPES = ["PD", "RJ", "RV"]

cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
coh_q1 = cohorts[(cohorts.prediction_month.astype(str) >= Q1_START) & (cohorts.prediction_month.astype(str) <= Q1_END)].copy()
assert list(coh_q1.cohort_id) == ["c19", "c20", "c21"] and (coh_q1.offset_days == 0).all(), coh_q1
df_coh = (spark.createDataFrame(coh_q1[["cohort_id", "prediction_month", "lookback_end_date"]].astype(str))
          .withColumn("MONTH", F.to_date("prediction_month"))
          .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date")).drop("prediction_month", "lookback_end_date"))

# Same universe as HCP_Target_Variable; placeholder IDs (null, '', '0') dropped as in the FTP / copay builds (QC 01 F1)
hcp_universe = (spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"))
                .filter(F.col("PROVIDER_ID").isNotNull() & ~F.trim(F.col("PROVIDER_ID")).isin("", "0")).distinct())
n_hcp = hcp_universe.count()
print(f"NBRx claims pull {DATA_START_NBRX}..{Q1_END} | cohorts {list(coh_q1.cohort_id)} | HCP universe {n_hcp:,}")
display(coh_q1)

In [0]:
# Same source and filters as HCP_Target_Variable, with CLAIM_TYPE kept and extended to RJ and RV
query = f'''
    SELECT DISTINCT A.PATIENT_ID, A.PROVIDER_ID, A.CLAIM_ID, A.SVC_DT, A.CLAIM_TYPE, B.PRODUCT_GROUP
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE CLAIM_TYPE IN ('PD', 'RJ', 'RV')
    AND SVC_DT >= '{DATA_START_NBRX}'
    AND SVC_DT <= '{Q1_END}'
    AND PRODUCT_GROUP = 'KERENDIA'
'''
claims = (Get_Data_Snowflakes(query)
          .withColumn("SVC_DT", F.to_date("SVC_DT"))
          .withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string")))
claims_pd = claims.toPandas()          # small (one product, 15 months); materialise once so Snowflake is hit once
claims = spark.createDataFrame(claims_pd)
print(f"Claims: {len(claims_pd):,}")
print(claims_pd.groupby("CLAIM_TYPE").agg(claims=("CLAIM_ID", "size"), patients=("PATIENT_ID", "nunique")))

In [0]:
# Rule from HCP_Target_Variable, partitioned additionally by CLAIM_TYPE so each type has its own patient history.
# The order includes CLAIM_ID so the rank-1 claim of a day is deterministic and gets the previous day as LAG.
keys = ["PATIENT_ID", "PRODUCT_GROUP", "CLAIM_TYPE"]
dedup_w = Window.partitionBy(*keys, "SVC_DT").orderBy("CLAIM_ID")
hist_w = Window.partitionBy(*keys).orderBy("SVC_DT", "CLAIM_ID")

claims_nbrx = (claims
    .withColumn("CLAIM_RANK", F.row_number().over(dedup_w))
    .withColumn("MIN_SVC_DT", F.min("SVC_DT").over(Window.partitionBy(*keys)))
    .withColumn("LAG_SVC_DT", F.lag("SVC_DT").over(hist_w))
    .withColumn("NBRX_FLAG",
        F.when((F.col("MIN_SVC_DT") == F.col("SVC_DT")) & (F.col("CLAIM_RANK") == 1), 1)
         .when(F.col("LAG_SVC_DT").isNotNull() & (F.col("LAG_SVC_DT") < F.date_add("SVC_DT", -365)) & (F.col("CLAIM_RANK") == 1), 1)
         .otherwise(0))
    .select("CLAIM_ID", "PROVIDER_ID", "SVC_DT", "CLAIM_TYPE", "NBRX_FLAG").distinct()
    .filter((F.col("SVC_DT") >= Q1_START) & (F.col("SVC_DT") <= Q1_END))
    .withColumn("MONTH", F.trunc("SVC_DT", "month")))

claims_nbrx.groupBy("CLAIM_TYPE", "MONTH").agg(F.count("*").alias("claims"), F.sum("NBRX_FLAG").alias("nbrx")).orderBy("CLAIM_TYPE", "MONTH").display()

In [0]:
agg = (claims_nbrx.groupBy("PROVIDER_ID", "MONTH").pivot("CLAIM_TYPE", CLAIM_TYPES).agg(F.sum("NBRX_FLAG"))
       .select("PROVIDER_ID", "MONTH", *[F.col(t).alias(f"KERENDIA_{t}_NBRX") for t in CLAIM_TYPES]))
NBRX_COLS = [f"KERENDIA_{t}_NBRX" for t in CLAIM_TYPES]

hcp_month = (hcp_universe.crossJoin(F.broadcast(df_coh))
    .join(agg, ["PROVIDER_ID", "MONTH"], "left").fillna(0, subset=NBRX_COLS)
    .select("PROVIDER_ID", "cohort_id", "MONTH", "LOOKBACK_END_M", *[F.col(c).cast("int").alias(c) for c in NBRX_COLS])
    .withColumn("NEW_WRITER", (F.col("KERENDIA_PD_NBRX") > 0).cast("int"))
    .withColumn("REJECTED_NEW_WRITER", ((F.col("KERENDIA_RJ_NBRX") > 0) | (F.col("KERENDIA_RV_NBRX") > 0)).cast("int")))

hcp_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH)
hm = spark.read.format("delta").load(OUTPUT_PATH)
hmp = hm.toPandas()
print(f"Saved {len(hmp):,} rows -> {OUTPUT_PATH}")
display(hmp.head())

In [0]:
import builtins   # the connection helper star-imports pyspark functions, which shadow round / max / min
# 1. Grid: exactly 3 rows per HCP, unique keys
assert len(hmp) == n_hcp * 3, (len(hmp), n_hcp * 3)
assert not hmp.duplicated(["PROVIDER_ID", "MONTH"]).any()
assert (hmp.groupby("PROVIDER_ID").size() == 3).all()

# 2. Totals reconcile with flagged claims of universe HCPs
flag_pd = claims_nbrx.join(hcp_universe, "PROVIDER_ID").groupBy("CLAIM_TYPE").agg(F.sum("NBRX_FLAG").alias("n")).toPandas().set_index("CLAIM_TYPE")["n"]
for t in CLAIM_TYPES:
    assert hmp[f"KERENDIA_{t}_NBRX"].sum() == flag_pd.get(t, 0), t

# 3. PD NBRx vs the saved target variable (KERENDIA_NBRX for c19..c21)
tv = (spark.read.format("delta").load(TARGET_PATH).filter(F.col("cohort_id").isin("c19", "c20", "c21"))
      .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"), "cohort_id", "KERENDIA_NBRX").toPandas())
cmp_tv = hmp.merge(tv, on=["PROVIDER_ID", "cohort_id"], how="left")
print("Target variable rows matched:", cmp_tv.KERENDIA_NBRX.notna().sum(), "of", len(cmp_tv))
print("Exact PD NBRx match vs target variable: ", builtins.round((cmp_tv.KERENDIA_PD_NBRX == cmp_tv.KERENDIA_NBRX).mean() * 100, 3), "%")
print(cmp_tv.groupby("cohort_id")[["KERENDIA_PD_NBRX", "KERENDIA_NBRX"]].sum())

# 4. Cross-check vs the precomputed monthly table (NBRX_KERENDIA_PD / RJ / RV)
mon = (spark.read.format("delta").load(MONTHLY_RX_PATH)
       .filter((F.col("SVC_DT_M") >= Q1_START) & (F.col("SVC_DT_M") <= Q1_END))
       .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"), F.to_date("SVC_DT_M").alias("MONTH"),
               "NBRX_KERENDIA_PD", "NBRX_KERENDIA_RJ", "NBRX_KERENDIA_RV").toPandas())
hmp["MONTH"] = pd.to_datetime(hmp.MONTH); mon["MONTH"] = pd.to_datetime(mon.MONTH)
cmp_m = hmp.merge(mon, on=["PROVIDER_ID", "MONTH"], how="left").fillna(0)
print("\nMonthly table rows in Q1 2026:", len(mon))
print(cmp_m.groupby("MONTH")[["KERENDIA_PD_NBRX", "NBRX_KERENDIA_PD", "KERENDIA_RJ_NBRX", "NBRX_KERENDIA_RJ", "KERENDIA_RV_NBRX", "NBRX_KERENDIA_RV"]].sum())

In [0]:
hmp["RJ_ONLY"] = ((hmp.KERENDIA_RJ_NBRX > 0) & (hmp.KERENDIA_RV_NBRX == 0)).astype(int)
hmp["RV_ONLY"] = ((hmp.KERENDIA_RV_NBRX > 0) & (hmp.KERENDIA_RJ_NBRX == 0)).astype(int)
hmp["RJ_AND_RV"] = ((hmp.KERENDIA_RJ_NBRX > 0) & (hmp.KERENDIA_RV_NBRX > 0)).astype(int)
hmp["REJ_AND_NEW"] = (hmp.REJECTED_NEW_WRITER & hmp.NEW_WRITER).astype(int)
hmp["REJ_ONLY_NO_PD"] = (hmp.REJECTED_NEW_WRITER & (1 - hmp.NEW_WRITER)).astype(int)

counts = hmp.groupby(hmp.MONTH.dt.strftime("%Y-%m")).agg(
    universe=("PROVIDER_ID", "nunique"),
    new_writers=("NEW_WRITER", "sum"), pd_nbrx=("KERENDIA_PD_NBRX", "sum"),
    rejected_new_writers=("REJECTED_NEW_WRITER", "sum"), rj_only=("RJ_ONLY", "sum"), rv_only=("RV_ONLY", "sum"),
    rj_and_rv=("RJ_AND_RV", "sum"), rejected_and_new_writer=("REJ_AND_NEW", "sum"), rejected_no_pd=("REJ_ONLY_NO_PD", "sum"),
    rj_nbrx=("KERENDIA_RJ_NBRX", "sum"), rv_nbrx=("KERENDIA_RV_NBRX", "sum"))
q = hmp.groupby("PROVIDER_ID")[["NEW_WRITER", "REJECTED_NEW_WRITER", "REJ_AND_NEW", "REJ_ONLY_NO_PD"]].max()
counts.loc["Q1 2026 (distinct HCPs)"] = [n_hcp, q.NEW_WRITER.sum(), hmp.KERENDIA_PD_NBRX.sum(), q.REJECTED_NEW_WRITER.sum(),
    np.nan, np.nan, np.nan, q.REJ_AND_NEW.sum(), q.REJ_ONLY_NO_PD.sum(), hmp.KERENDIA_RJ_NBRX.sum(), hmp.KERENDIA_RV_NBRX.sum()]
counts["new_writer_pct"] = (counts.new_writers / counts.universe * 100).round(2)
counts["rejected_pct"] = (counts.rejected_new_writers / counts.universe * 100).round(2)
display(counts.reset_index().rename(columns={"MONTH": "month"}))

In [0]:
# Table 1: Paid Writers (PD NBRx > 0) and Paid NBRx. Table 2: Non-paid Writers (RJ or RV NBRx > 0) and Non-paid NBRx (RJ + RV),
# split into Type 1 (paid and non-paid NBRx in the same month) and Type 2 (only non-paid NBRx in the month).
# Q1 writer rows count distinct HCPs; Q1 NBRx rows sum the HCP-months, so Type 1 + Type 2 NBRx = Non-paid NBRx.
l = hmp.copy()
l["M"] = l.MONTH.dt.strftime("%Y-%m")
l["NONPAID_NBRX"] = l.KERENDIA_RJ_NBRX + l.KERENDIA_RV_NBRX
l["TYPE1"] = ((l.NONPAID_NBRX > 0) & (l.KERENDIA_PD_NBRX > 0)).astype(int)
l["TYPE2"] = ((l.NONPAID_NBRX > 0) & (l.KERENDIA_PD_NBRX == 0)).astype(int)
l["T1_NBRX"] = l.NONPAID_NBRX * l.TYPE1
l["T2_NBRX"] = l.NONPAID_NBRX * l.TYPE2
l["T1_PD_NBRX"] = l.KERENDIA_PD_NBRX * l.TYPE1          # reference only: paid NBRx written by Type 1 writers

def q1_row(d, flags, sums):
    hcp = d.groupby("PROVIDER_ID")[flags].max().sum()
    return pd.concat([hcp, d[sums].sum()])

paid_cols = ["NEW_WRITER", "KERENDIA_PD_NBRX"]
t_paid = l.groupby("M")[paid_cols].sum()
t_paid.loc["Q1 distinct HCPs"] = q1_row(l, ["NEW_WRITER"], ["KERENDIA_PD_NBRX"])[paid_cols]
t_paid.columns = ["Paid Writers", "Paid NBRx"]

np_cols = ["REJECTED_NEW_WRITER", "NONPAID_NBRX", "TYPE1", "T1_NBRX", "TYPE2", "T2_NBRX", "T1_PD_NBRX"]
t_np = l.groupby("M")[np_cols].sum()
t_np.loc["Q1 distinct HCPs"] = q1_row(l, ["REJECTED_NEW_WRITER", "TYPE1", "TYPE2"], ["NONPAID_NBRX", "T1_NBRX", "T2_NBRX", "T1_PD_NBRX"])[np_cols]
t_np.columns = ["Non-paid Writers", "Non-paid NBRx", "Type 1 Writers", "Type 1 Non-paid NBRx", "Type 2 Writers", "Type 2 Non-paid NBRx", "Type 1 Paid NBRx (ref)"]
t_paid, t_np = t_paid.astype(int), t_np.astype(int)

# Checks: match the Steps 2-3 table, and the split is complete
assert (t_paid["Paid Writers"] == counts.new_writers.astype(int).values).all()
assert (t_np["Non-paid Writers"] == counts.rejected_new_writers.astype(int).values).all()
assert (t_np["Type 1 Writers"] == counts.rejected_and_new_writer.astype(int).values).all()
assert (t_np["Type 2 Writers"] == counts.rejected_no_pd.astype(int).values).all()
assert (t_np["Type 1 Non-paid NBRx"] + t_np["Type 2 Non-paid NBRx"] == t_np["Non-paid NBRx"]).all()
assert (t_np["Non-paid NBRx"] == (counts.rj_nbrx + counts.rv_nbrx).astype(int).values).all()
print("Checks passed")
display(t_paid.reset_index().rename(columns={"M": "Month", "index": "Month"}))
display(t_np.reset_index().rename(columns={"M": "Month", "index": "Month"}))

In [0]:
KEY = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT", "PULL_DT"]
ftp = spark.read.format("delta").load(FTP_PATH).filter(F.col("COHORT").isin("c19", "c20", "c21"))
cpy = spark.read.format("delta").load(CPY_PATH).filter(F.col("COHORT").isin("c19", "c20", "c21"))
FTP_FEATS = [c for c in ftp.columns if c not in KEY]
CPY_FEATS = [c for c in cpy.columns if c not in KEY]
FEATS = FTP_FEATS + CPY_FEATS

feat = (ftp.select("PROVIDER_ID", F.col("COHORT").alias("cohort_id"), *FTP_FEATS)
        .join(cpy.select("PROVIDER_ID", F.col("COHORT").alias("cohort_id"), *CPY_FEATS), ["PROVIDER_ID", "cohort_id"], "outer"))
pw = (spark.read.format("delta").load(PRIOR_WRITING_PATH).filter(F.col("cohort_id").isin("c19", "c20", "c21"))
      .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"), "cohort_id", "KERENDIA_NBRX_IN_LAST_12_MONTH", "KERENDIA_TRX_IN_LAST_12_MONTH"))
fp = feat.join(pw, ["PROVIDER_ID", "cohort_id"], "outer").toPandas()

df = hmp.merge(fp, on=["PROVIDER_ID", "cohort_id"], how="left")
assert len(df) == len(hmp)
print(f"FTP feats {len(FTP_FEATS)}, copay feats {len(CPY_FEATS)}; HCP-months without FTP/copay row: {df[FTP_FEATS[0]].isna().sum():,}; without prior writing: {df.KERENDIA_TRX_IN_LAST_12_MONTH.isna().sum():,}")
df["PRIOR_WRITER_12M"] = (df.KERENDIA_TRX_IN_LAST_12_MONTH.fillna(0) > 0).astype(int)
df["GROUP"] = np.select([df.REJ_ONLY_NO_PD == 1, df.REJ_AND_NEW == 1, df.NEW_WRITER == 1],
                        ["Rejected only (no PD)", "Rejected + New Writer", "New Writer only"], "Neither")
display(df.groupby(["cohort_id", "GROUP"]).size().unstack())

In [0]:
from scipy import stats
from sklearn.metrics import roc_auc_score

def bh_adjust(p):
    """Benjamini-Hochberg FDR-adjusted p-values (statsmodels is not installed on this compute)."""
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    adj = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(adj, 1.0)
    return out

def signal_table(d, label, feats):
    """Per feature: non-zero share, mean, median by group, SMD, AUC (oriented), Mann-Whitney p (BH-adjusted), top-decile lift."""
    y = d[label].values; base = y.mean(); rows = []
    for f in feats:
        x = d[f].astype(float)
        if f.endswith("DAYS_SINCE_LAST_FILL"):
            x = x.fillna(9999)
        x = x.fillna(0).values
        a, b = x[y == 1], x[y == 0]
        sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if len(a) > 1 else np.nan
        smd = (a.mean() - b.mean()) / sd if sd and sd > 0 else 0.0
        auc = roc_auc_score(y, x) if np.unique(x).size > 1 else 0.5
        p = stats.mannwhitneyu(a, b).pvalue if np.unique(x).size > 1 else 1.0
        o = x if auc >= 0.5 else -x
        top = o >= np.quantile(o, 0.9)
        lift = y[top].mean() / base if top.any() and base > 0 else np.nan
        rows.append(dict(feature=f, family="FTP" if f.startswith("FTP_") and f != "FTP_CPY_BOTH_12M" else "Copay",
                         nonzero_pct_pos=(a != 0).mean() * 100, nonzero_pct_rest=(b != 0).mean() * 100,
                         mean_pos=a.mean(), mean_rest=b.mean(), median_pos=np.median(a), median_rest=np.median(b),
                         smd=smd, auc=builtins.max(auc, 1 - auc), direction="higher" if auc >= 0.5 else "lower", p=p, top10_lift=lift))
    t = pd.DataFrame(rows)
    t["p_bh"] = bh_adjust(t.p)
    t["verdict"] = np.select([(t.smd.abs() >= 0.2) & (t.auc >= 0.6) & (t.p_bh < 0.05), (t.p_bh < 0.05) & ((t.smd.abs() >= 0.1) | (t.auc >= 0.55))],
                             ["strong", "weak"], "none")
    return t.sort_values("auc", ascending=False).round(4)

print(f"Positives: {df.REJECTED_NEW_WRITER.sum():,} of {len(df):,} HCP-months (Q1 2026 pooled)")
sig_all = signal_table(df, "REJECTED_NEW_WRITER", FEATS)
display(sig_all)

In [0]:
res = {}
for c in ["c19", "c20", "c21"]:
    res[f"auc_{c}"] = signal_table(df[df.cohort_id == c], "REJECTED_NEW_WRITER", FEATS).set_index("feature").auc
# Same comparison restricted to HCPs with any Kerendia TRx in the prior 12 months (removes the writer / non-writer effect)
pw_df = df[df.PRIOR_WRITER_12M == 1]
print(f"Prior writers: {len(pw_df):,} HCP-months, positives {pw_df.REJECTED_NEW_WRITER.sum():,}")
sig_pw = signal_table(pw_df, "REJECTED_NEW_WRITER", FEATS).set_index("feature")
res["auc_prior_writers"], res["smd_prior_writers"], res["verdict_prior_writers"] = sig_pw.auc, sig_pw.smd, sig_pw.verdict
# Rejected with no paid NBRx vs New Writers only: do features separate access failure from success?
rv = df[df.GROUP.isin(["Rejected only (no PD)", "New Writer only"])].assign(Y=lambda d: (d.GROUP == "Rejected only (no PD)").astype(int))
print(f"Rejected-only vs New-Writer-only: {len(rv):,} HCP-months, positives {rv.Y.sum():,}")
sig_rv = signal_table(rv, "Y", FEATS).set_index("feature")
res["auc_rej_vs_new"], res["smd_rej_vs_new"], res["dir_rej_vs_new"], res["verdict_rej_vs_new"] = sig_rv.auc, sig_rv.smd, sig_rv.direction, sig_rv.verdict
robust = sig_all.set_index("feature")[["family", "auc", "smd", "verdict"]].join(pd.DataFrame(res)).sort_values("auc", ascending=False).round(3)
display(robust.reset_index())

In [0]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier

def cv_auc(d, label, feats):
    X = d[feats].astype(float).copy()
    for f in feats:
        if f.endswith("DAYS_SINCE_LAST_FILL"):
            X[f] = X[f].fillna(9999)
    X = X.fillna(0)
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0)
    p = cross_val_predict(clf, X, d[label], cv=GroupKFold(5), groups=d.PROVIDER_ID, method="predict_proba")[:, 1]   # HCP-grouped folds
    return roc_auc_score(d[label], p)

mv = []
for name, d, lab in [("Rejected vs rest (all)", df, "REJECTED_NEW_WRITER"), ("Rejected vs rest (prior writers)", pw_df, "REJECTED_NEW_WRITER"),
                     ("Rejected-only vs New-Writer-only", rv, "Y")]:
    mv.append(dict(comparison=name, n=len(d), positives=int(d[lab].sum()),
                   auc_ftp=cv_auc(d, lab, FTP_FEATS), auc_copay=cv_auc(d, lab, CPY_FEATS), auc_ftp_copay=cv_auc(d, lab, FEATS),
                   auc_prior_writing_only=cv_auc(d, lab, ["KERENDIA_NBRX_IN_LAST_12_MONTH", "KERENDIA_TRX_IN_LAST_12_MONTH"]),
                   auc_prior_writing_plus_ftp_copay=cv_auc(d, lab, FEATS + ["KERENDIA_NBRX_IN_LAST_12_MONTH", "KERENDIA_TRX_IN_LAST_12_MONTH"])))
display(pd.DataFrame(mv).round(4))

In [0]:
# Recompute the NBRx flag in pandas (no Spark windows) from the raw claims and recount writers per month
c = claims_pd.copy(); c["SVC_DT"] = pd.to_datetime(c.SVC_DT); c["PROVIDER_ID"] = c.PROVIDER_ID.astype(str)
c = c.sort_values(["PATIENT_ID", "CLAIM_TYPE", "SVC_DT", "CLAIM_ID"])
first_of_day = ~c.duplicated(["PATIENT_ID", "CLAIM_TYPE", "SVC_DT"])
prev = c.groupby(["PATIENT_ID", "CLAIM_TYPE"]).SVC_DT.shift()
c["NEW"] = first_of_day & (prev.isna() | (prev < c.SVC_DT - pd.Timedelta(days=365)))
uni = set(hmp.PROVIDER_ID)
q1 = c[c.NEW & (c.SVC_DT >= Q1_START) & (c.SVC_DT <= Q1_END) & c.PROVIDER_ID.isin(uni)].assign(M=lambda d: d.SVC_DT.dt.strftime("%Y-%m"))
chk = pd.DataFrame({
    "new_writers_pandas": q1[q1.CLAIM_TYPE == "PD"].groupby("M").PROVIDER_ID.nunique(),
    "rejected_new_writers_pandas": q1[q1.CLAIM_TYPE.isin(["RJ", "RV"])].groupby("M").PROVIDER_ID.nunique()})
chk = chk.join(counts[["new_writers", "rejected_new_writers"]])
chk["match"] = (chk.new_writers_pandas == chk.new_writers) & (chk.rejected_new_writers_pandas == chk.rejected_new_writers)
display(chk.reset_index().rename(columns={"index": "month"}))

## Findings (run 25 Nov 2026)

HCP universe 294,726. Tables from Step 3b.

**Paid Writers (PD NBRx > 0)**

| Month | Paid Writers | Paid NBRx |
| --- | --- | --- |
| Jan-26 | 5,625 | 8,138 |
| Feb-26 | 5,877 | 8,693 |
| Mar-26 | 6,194 | 9,290 |
| Q1 distinct HCPs | 12,815 | 26,121 |

**Non-paid Writers (RJ or RV NBRx > 0)**

| Month | Non-paid Writers | Non-paid NBRx | Type 1 Writers (mix of Paid and Non-paid NBRx) | Type 1 Writers NBRx | Type 2 Writers (only Non-paid NBRx) | Type 2 Writers NBRx |
| --- | --- | --- | --- | --- | --- | --- |
| Jan-26 | 8,521 | 14,431 | 3,097 | 7,261 | 5,424 | 7,170 |
| Feb-26 | 7,833 | 12,759 | 3,052 | 6,644 | 4,781 | 6,115 |
| Mar-26 | 8,069 | 12,990 | 3,264 | 6,975 | 4,805 | 6,015 |
| Q1 distinct HCPs | 16,723 | 40,180 | 6,724 | 20,880 | 12,736 | 19,300 |

* **Definitions:** Non-paid NBRx = RJ + RV NBRx. Type 1 and Type 2 are set per month; Type 1 and Type 2 NBRx columns are their Non-paid NBRx, so Type 1 + Type 2 = Non-paid NBRx. NBRx in the Q1 row is the Q1 total. Writer counts in the Q1 row are distinct HCPs, and an HCP can be Type 1 in one month and Type 2 in another, so 6,724 + 12,736 is more than 16,723.
* **For reference:** Type 1 writers also wrote 5,255 / 5,425 / 5,883 Paid NBRx in Jan / Feb / Mar (16,563 in Q1), about 63% of all Paid NBRx.
* **Type 2 writers average fewer non-paid NBRx:** about 1.3 per writer per month vs 2.1 to 2.3 for Type 1.
* **Checks:** PD NBRx matches `HCP_Target_Variable` totals for c19 to c21 (8,138 / 8,693 / 9,290), with 99.987% of rows matching exactly. The writer counts match a separate pandas recount. The older monthly table `kerendia_claims_monthly_from_raw_fct_rx` gives lower NBRx (PD 5,245 for Jan), so it must use a different rule.
* **Rejected vs rest:** the strongest single features are `FTP_DAYS_SINCE_LAST_FILL` (AUC 0.70) and `CPY_DAYS_SINCE_LAST_FILL` (AUC 0.64). All other features score 0.60 or lower (copay 12M volume features about 0.596, FTP 12M about 0.59). The AUCs are about the same in c19, c20 and c21.
* **This is mostly the writer effect:** 21,548 of the 24,423 rejected HCP-months come from HCPs with Kerendia TRx in the prior 12 months. Prior writing alone gives a cross-validated AUC of 0.909. FTP + copay alone give 0.725, and adding them to prior writing raises it only to 0.910 (prior writers only: 0.827 -> 0.827).
* **Rejected only vs New Writers only:** every feature has AUC 0.54 or lower, and FTP + copay together give 0.546. These features cannot tell an HCP whose new patient is rejected apart from one who gets a paid claim.
* **No signal:** `FTP_OTHER_CVRG_REJECTED_SHARE_12M` (expected: code 03 is absent after 2024), `CPY_NO_OTHER_CVRG_SHARE_12M` and `FTP_PAT_GENDER_U_SHARE_12M` show AUC 0.50 and SMD 0 in every comparison.

## BlinkRx Analysis

Same writer table as the LAAD steps above, built only from BlinkRx data (`CPH_DB_PROD.ENRICH.BLINKRX_DISPENSE`), for Jan to Mar 2026 on the same HCP universe.

* **NBRx rule (from `02_Blink_Rx_NBRx`):** `SCRIPT_TYPE = 'new to Blink'`, valid prescriber NPI. Per patient and product, a script is NBRx if it is the patient's first BlinkRx script or comes 12+ months after the previous one, and only the first NBRx per patient and product is kept. Its `COVERAGE_CODE` labels the NBRx; null coverage becomes `ABANDON`.
* **Paid NBRx** = coverage `P`. **Non-paid NBRx** = coverage `R` or `ABANDON`.
* **BlinkRx Paid Writers** = Paid NBRx > 0 in the month. **BlinkRx Non-paid Writers** = R or ABANDON NBRx > 0 in the month.
* Prescriber NPI is mapped to `PROVIDER_ID` with `kerendia_nba_3_0_npi_bhid_mapping`.

In [0]:
# Source and filters exactly as in 02_Blink_Rx_NBRx. No date filter: the NBRx rule needs the full patient history (BlinkRx starts Mar 2024).
NPI_MAP_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/query_output/kerendia_nba_3_0_npi_bhid_mapping"
BLINK_OUTPUT_PATH = f"{VOL31}/notebook_output/target_variable/ablation_q1_2026/hcp_month_blinkrx_nbrx_p_r_abandon"
BLINK_CODES = ["P", "R", "ABANDON"]

blink_query = '''
    SELECT DISTINCT PRESCRIBER_NPI, UNIQUE_PATIENT_ID, PRODUCT_GROUP, RX_RECEIVED_DATE, COVERAGE_CODE, SCRIPT_TYPE
    FROM CPH_DB_PROD.ENRICH.BLINKRX_DISPENSE A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    ON A.PRODUCT_NDC = B.NDC_CD
'''
blink_raw = Get_Data_Snowflakes(blink_query)
blink = (blink_raw.filter(F.col("SCRIPT_TYPE") == "new to Blink")
         .filter((F.col("PRESCRIBER_NPI") != "HIPAA Restricted") & F.col("PRESCRIBER_NPI").isNotNull())
         .withColumn("svc_dt", F.to_date("RX_RECEIVED_DATE"))
         .select(F.col("PRESCRIBER_NPI").cast("string").alias("PRESCRIBER_NPI"), "UNIQUE_PATIENT_ID", "PRODUCT_GROUP", "svc_dt", "COVERAGE_CODE"))
blink_pd = blink.toPandas()
print(f"New-to-Blink scripts (valid NPI): {len(blink_pd):,} | patients {blink_pd.UNIQUE_PATIENT_ID.nunique():,} | "
      f"products {sorted(blink_pd.PRODUCT_GROUP.unique())} | dates {blink_pd.svc_dt.min()}..{blink_pd.svc_dt.max()}")
print(blink_pd.COVERAGE_CODE.fillna("null").value_counts().to_string())

In [0]:
# 02_Blink_Rx_NBRx orders by svc_dt only. COVERAGE_CODE (P, R, null) is added as a tie-breaker so same-day scripts give a deterministic result.
w_blink = Window.partitionBy("UNIQUE_PATIENT_ID", "PRODUCT_GROUP").orderBy("svc_dt", F.col("COVERAGE_CODE").asc_nulls_last())
blink_nbrx = (blink.withColumn("prev_svc_dt", F.lag("svc_dt").over(w_blink))
    .withColumn("NBRx_flag", F.when(F.col("prev_svc_dt").isNull() | (F.months_between("svc_dt", "prev_svc_dt") >= 12), 1).otherwise(0))
    .filter(F.col("NBRx_flag") == 1)
    .withColumn("rn", F.row_number().over(w_blink)).filter(F.col("rn") == 1).drop("rn")     # first NBRx per patient x product
    .withColumn("COVERAGE_CODE", F.coalesce(F.col("COVERAGE_CODE"), F.lit("ABANDON")))     # same fill as 02 (done before aggregation, same result)
    .withColumn("MONTH", F.trunc("svc_dt", "month")))

blink_q1 = blink_nbrx.filter((F.col("svc_dt") >= Q1_START) & (F.col("svc_dt") <= Q1_END)).withColumnRenamed("PRESCRIBER_NPI", "NPI_NUMBER")
blink_agg_npi = blink_q1.groupBy("NPI_NUMBER", "MONTH").pivot("COVERAGE_CODE", BLINK_CODES).count().fillna(0)

# NPI -> PROVIDER_ID within the HCP universe. 4 NPIs map to 2 universe IDs; keep one so NBRx is not double counted.
npi_map = (spark.read.format("delta").load(NPI_MAP_PATH)
           .select(F.col("NPI_NUMBER").cast("string").alias("NPI_NUMBER"), F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"))
           .filter(F.col("NPI_NUMBER").isNotNull() & F.col("PROVIDER_ID").isNotNull()).distinct()
           .join(hcp_universe, "PROVIDER_ID").groupBy("NPI_NUMBER").agg(F.min("PROVIDER_ID").alias("PROVIDER_ID")))
blink_agg = (blink_agg_npi.join(npi_map, "NPI_NUMBER").groupBy("PROVIDER_ID", "MONTH")
             .agg(*[F.sum(c).alias(f"BLINK_{c}_NBRX") for c in BLINK_CODES]))
BLINK_COLS = [f"BLINK_{c}_NBRX" for c in BLINK_CODES]

blink_hm = (hcp_universe.crossJoin(F.broadcast(df_coh))
    .join(blink_agg, ["PROVIDER_ID", "MONTH"], "left").fillna(0, subset=BLINK_COLS)
    .select("PROVIDER_ID", "cohort_id", "MONTH", *[F.col(c).cast("int").alias(c) for c in BLINK_COLS])
    .withColumn("BLINK_PAID_WRITER", (F.col("BLINK_P_NBRX") > 0).cast("int"))
    .withColumn("BLINK_NONPAID_WRITER", ((F.col("BLINK_R_NBRX") > 0) | (F.col("BLINK_ABANDON_NBRX") > 0)).cast("int")))

blink_hm.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(BLINK_OUTPUT_PATH)
blink_hmp = spark.read.format("delta").load(BLINK_OUTPUT_PATH).toPandas()
blink_npi_pd = blink_agg_npi.toPandas()
print(f"Saved {len(blink_hmp):,} rows -> {BLINK_OUTPUT_PATH}")
display(blink_hmp[blink_hmp[BLINK_COLS].sum(axis=1) > 0].head())

In [0]:
# 1. Grid: exactly 3 rows per HCP, unique keys
assert len(blink_hmp) == n_hcp * 3, (len(blink_hmp), n_hcp * 3)
assert not blink_hmp.duplicated(["PROVIDER_ID", "MONTH"]).any()
# 2. Q1 BlinkRx NBRx before mapping vs inside the universe, by coverage code
mapped_npis = set(npi_map.select("NPI_NUMBER").toPandas().NPI_NUMBER)
cov = pd.DataFrame({"all_npis": blink_npi_pd[BLINK_CODES].sum(),
                    "npis_in_universe": blink_npi_pd[blink_npi_pd.NPI_NUMBER.isin(mapped_npis)][BLINK_CODES].sum(),
                    "grid_total": blink_hmp[BLINK_COLS].sum().values})
cov["pct_in_universe"] = (cov.npis_in_universe / cov.all_npis * 100).round(2)
assert (cov.npis_in_universe == cov.grid_total).all(), cov
print(f"Q1 BlinkRx prescriber NPIs: {blink_npi_pd.NPI_NUMBER.nunique():,}, of which in universe: {blink_npi_pd.NPI_NUMBER.isin(mapped_npis).groupby(blink_npi_pd.NPI_NUMBER).first().sum():,}")
display(cov.reset_index().rename(columns={"index": "COVERAGE_CODE"}))
# 3. Cross-check with the production BlinkRx table (BH_ID level, its own refresh date and mapping, so small gaps are expected)
prod = (spark.table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.blink_kerendia_nbrx_rj_rv_pd_inference")
        .filter((F.col("month") >= Q1_START) & (F.col("month") <= Q1_END))
        .groupBy(F.date_format("month", "yyyy-MM").alias("month")).pivot("COVERAGE_CODE", BLINK_CODES).agg(F.sum("NBRx")).toPandas().set_index("month").sort_index())
ours = blink_npi_pd.assign(month=pd.to_datetime(blink_npi_pd.MONTH).dt.strftime("%Y-%m")).groupby("month")[BLINK_CODES].sum()
display(ours.add_suffix("_this_notebook").join(prod.add_suffix("_prod_inference")).reset_index())

In [0]:
b = blink_hmp
b["R_ONLY"] = ((b.BLINK_R_NBRX > 0) & (b.BLINK_ABANDON_NBRX == 0)).astype(int)
b["ABANDON_ONLY"] = ((b.BLINK_ABANDON_NBRX > 0) & (b.BLINK_R_NBRX == 0)).astype(int)
b["R_AND_ABANDON"] = ((b.BLINK_R_NBRX > 0) & (b.BLINK_ABANDON_NBRX > 0)).astype(int)
b["NONPAID_AND_PAID"] = (b.BLINK_NONPAID_WRITER & b.BLINK_PAID_WRITER).astype(int)
b["NONPAID_NO_PAID"] = (b.BLINK_NONPAID_WRITER & (1 - b.BLINK_PAID_WRITER)).astype(int)

blink_counts = b.groupby(pd.to_datetime(b.MONTH).dt.strftime("%Y-%m")).agg(
    universe=("PROVIDER_ID", "nunique"),
    paid_writers=("BLINK_PAID_WRITER", "sum"), p_nbrx=("BLINK_P_NBRX", "sum"),
    nonpaid_writers=("BLINK_NONPAID_WRITER", "sum"), r_only=("R_ONLY", "sum"), abandon_only=("ABANDON_ONLY", "sum"),
    r_and_abandon=("R_AND_ABANDON", "sum"), nonpaid_and_paid=("NONPAID_AND_PAID", "sum"), nonpaid_no_paid=("NONPAID_NO_PAID", "sum"),
    r_nbrx=("BLINK_R_NBRX", "sum"), abandon_nbrx=("BLINK_ABANDON_NBRX", "sum"))
bq = b.groupby("PROVIDER_ID")[["BLINK_PAID_WRITER", "BLINK_NONPAID_WRITER", "NONPAID_AND_PAID", "NONPAID_NO_PAID"]].max()
blink_counts.loc["Q1 2026 (distinct HCPs)"] = [n_hcp, bq.BLINK_PAID_WRITER.sum(), b.BLINK_P_NBRX.sum(), bq.BLINK_NONPAID_WRITER.sum(),
    np.nan, np.nan, np.nan, bq.NONPAID_AND_PAID.sum(), bq.NONPAID_NO_PAID.sum(), b.BLINK_R_NBRX.sum(), b.BLINK_ABANDON_NBRX.sum()]
blink_counts["paid_writer_pct"] = (blink_counts.paid_writers / blink_counts.universe * 100).round(2)
blink_counts["nonpaid_writer_pct"] = (blink_counts.nonpaid_writers / blink_counts.universe * 100).round(2)
display(blink_counts.reset_index().rename(columns={"MONTH": "month"}))

In [0]:
# Recompute the BlinkRx NBRx in pandas (no Spark windows) from the raw scripts and recount writers per month
c = blink_pd.copy(); c["svc_dt"] = pd.to_datetime(c.svc_dt)
c["CODE"] = c.COVERAGE_CODE.fillna("ABANDON")
c["ORD"] = c.COVERAGE_CODE.map({"P": 0, "R": 1}).fillna(2)          # same tie-break as Step B2: P, R, null
c = c.sort_values(["UNIQUE_PATIENT_ID", "PRODUCT_GROUP", "svc_dt", "ORD"])
prev = c.groupby(["UNIQUE_PATIENT_ID", "PRODUCT_GROUP"]).svc_dt.shift()
c["NEW"] = prev.isna() | (c.svc_dt >= prev + pd.DateOffset(months=12))
first_new = c[c.NEW].drop_duplicates(["UNIQUE_PATIENT_ID", "PRODUCT_GROUP"])
nmap = npi_map.toPandas().set_index("NPI_NUMBER").PROVIDER_ID
q1b = first_new[(first_new.svc_dt >= Q1_START) & (first_new.svc_dt <= Q1_END)].assign(
    PROVIDER_ID=lambda d: d.PRESCRIBER_NPI.map(nmap), M=lambda d: d.svc_dt.dt.strftime("%Y-%m")).dropna(subset=["PROVIDER_ID"])
bchk = pd.DataFrame({
    "paid_writers_pandas": q1b[q1b.CODE == "P"].groupby("M").PROVIDER_ID.nunique(),
    "nonpaid_writers_pandas": q1b[q1b.CODE.isin(["R", "ABANDON"])].groupby("M").PROVIDER_ID.nunique()})
bchk = bchk.join(blink_counts[["paid_writers", "nonpaid_writers"]])
bchk["match"] = (bchk.paid_writers_pandas == bchk.paid_writers) & (bchk.nonpaid_writers_pandas == bchk.nonpaid_writers)
display(bchk.reset_index().rename(columns={"index": "month"}))

## BlinkRx Findings (run 25 Nov 2026)

BlinkRx NBRx only (`02_Blink_Rx_NBRx` rule), HCP universe of 294,726, Jan to Mar 2026. Output: `ablation_q1_2026/hcp_month_blinkrx_nbrx_p_r_abandon` (884,178 rows).

| Month | Paid Writers (P > 0) | Paid NBRx | Non-paid Writers (R or ABANDON > 0) | R only | ABANDON only | R and ABANDON | Non-paid and Paid | Non-paid, no Paid | R NBRx | ABANDON NBRx |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-01 | 89 | 114 | 100 | 75 | 22 | 3 | 21 | 79 | 109 | 27 |
| 2026-02 | 142 | 171 | 112 | 71 | 29 | 12 | 25 | 87 | 105 | 46 |
| 2026-03 | 173 | 208 | 130 | 97 | 26 | 7 | 33 | 97 | 120 | 36 |
| Q1 distinct HCPs | 353 | 493 | 291 | | | | 65 | 246 | 334 | 109 |

* **Scale:** BlinkRx is very small. 6,200 new-to-Blink scripts in total (6,153 patients, Kerendia only, Mar 2024 to Nov 2026). In Q1 2026 it gives 936 NBRx in the universe. Paid Writers are 0.03 to 0.06% of the universe per month and Non-paid Writers 0.03 to 0.04%. LAAD New Writers for the same quarter are 12,815.
* **Paid share:** 493 of 936 Q1 NBRx (53%) are paid. Of the 443 non-paid, 334 are `R` and 109 are `ABANDON` (null coverage filled in).
* **Non-paid usually means no paid:** 246 of the 291 Non-paid Writers (85%) have no paid BlinkRx NBRx in Q1. Only 65 HCPs have both.
* **Paid writers are growing:** 89 -> 142 -> 173 from Jan to Mar. Non-paid writers grow less (100 -> 112 -> 130).
* **Checks:** 552 of the 555 Q1 prescriber NPIs map to the universe (3 paid NBRx are left out). Grid totals equal the mapped NBRx for each coverage code. Monthly totals match the production table `blink_kerendia_nbrx_rj_rv_pd_inference` exactly, except Feb paid (172 vs 171). A separate pandas recount gives the same writer counts in all three months.
* **Method notes:** unlike the LAAD steps, BlinkRx gives each patient one NBRx (its first new script), labelled by that script's coverage. So a patient whose script was rejected and later paid counts as non-paid only. 6 patients have more than one coverage code on their first date; ties are broken P, then R, then null.

In [0]:
# LAAD Paid Writers = NEW_WRITER (PD NBRx > 0); LAAD Non-paid Writers = REJECTED_NEW_WRITER (RJ or RV NBRx > 0), from Steps 1c-3.
# BlinkRx Paid Writers = BLINK_PAID_WRITER (P > 0); BlinkRx Non-paid Writers = BLINK_NONPAID_WRITER (R or ABANDON > 0), from Step B2.
# MONTH is datetime in hmp and a date object in blink_hmp, so join on a YYYY-MM string
laad_w = hmp[["PROVIDER_ID", "NEW_WRITER", "REJECTED_NEW_WRITER"]].assign(M=pd.to_datetime(hmp.MONTH).dt.strftime("%Y-%m"))
blink_w = blink_hmp[["PROVIDER_ID", "BLINK_PAID_WRITER", "BLINK_NONPAID_WRITER"]].assign(M=pd.to_datetime(blink_hmp.MONTH).dt.strftime("%Y-%m"))
ov = laad_w.merge(blink_w, on=["PROVIDER_ID", "M"], how="inner", validate="1:1")
assert len(ov) == len(hmp) == len(blink_hmp)

def overlap(blink_col, laad_col, label):
    rows = []
    groups = [(m, g) for m, g in ov.groupby("M")] + [("Q1 2026 (distinct HCPs)", ov.groupby("PROVIDER_ID")[[blink_col, laad_col]].max())]
    for m, g in groups:
        bl, la = g[blink_col] == 1, g[laad_col] == 1
        both = int((bl & la).sum())
        rows.append(dict(comparison=label, month=m, blinkrx_writers=int(bl.sum()), laad_writers=int(la.sum()), both=both,
                         blinkrx_only=int((bl & ~la).sum()), laad_only=int((la & ~bl).sum()),
                         pct_blinkrx_in_laad=builtins.round(both / bl.sum() * 100, 1) if bl.sum() else np.nan,   # builtins: pyspark star import shadows round
                         pct_laad_in_blinkrx=builtins.round(both / la.sum() * 100, 2) if la.sum() else np.nan,
                         jaccard_pct=builtins.round(both / (bl | la).sum() * 100, 2)))
    return pd.DataFrame(rows)

overlap_tbl = pd.concat([overlap("BLINK_PAID_WRITER", "NEW_WRITER", "Paid: BlinkRx P vs LAAD PD"),
                         overlap("BLINK_NONPAID_WRITER", "REJECTED_NEW_WRITER", "Non-paid: BlinkRx R/ABANDON vs LAAD RJ/RV")], ignore_index=True)

# Independent check: set intersections of PROVIDER_IDs per month, straight from the two source tables
for (bc, lc, lab) in [("BLINK_PAID_WRITER", "NEW_WRITER", "Paid"), ("BLINK_NONPAID_WRITER", "REJECTED_NEW_WRITER", "Non-paid")]:
    for m in sorted(ov.M.unique()):
        s_b = set(blink_w.loc[(blink_w.M == m) & (blink_w[bc] == 1), "PROVIDER_ID"])
        s_l = set(laad_w.loc[(laad_w.M == m) & (laad_w[lc] == 1), "PROVIDER_ID"])
        r = overlap_tbl[overlap_tbl.comparison.str.startswith(lab) & (overlap_tbl.month == m)].iloc[0]
        assert (len(s_b), len(s_l), len(s_b & s_l)) == (r.blinkrx_writers, r.laad_writers, r.both), (lab, m)
print("Set-intersection check passed for all months")
display(overlap_tbl)

## BlinkRx vs LAAD Writer Overlap (run 25 Nov 2026)

Same HCP universe (294,726) and HCP-month grid on both sides. Paid: BlinkRx `P` NBRx > 0 vs LAAD PD NBRx > 0 (New Writers). Non-paid: BlinkRx `R` or `ABANDON` NBRx > 0 vs LAAD RJ or RV NBRx > 0 (Rejected New Writers).

**Paid Writers**

| Month | BlinkRx | LAAD | Both | BlinkRx only | LAAD only | % of BlinkRx also in LAAD | % of LAAD also in BlinkRx |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-01 | 89 | 5,625 | 44 | 45 | 5,581 | 49.4% | 0.78% |
| 2026-02 | 142 | 5,877 | 62 | 80 | 5,815 | 43.7% | 1.05% |
| 2026-03 | 173 | 6,194 | 95 | 78 | 6,099 | 54.9% | 1.53% |
| Q1 distinct HCPs | 353 | 12,815 | 262 | 91 | 12,553 | 74.2% | 2.04% |

**Non-paid Writers**

| Month | BlinkRx | LAAD | Both | BlinkRx only | LAAD only | % of BlinkRx also in LAAD | % of LAAD also in BlinkRx |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-01 | 100 | 8,521 | 62 | 38 | 8,459 | 62.0% | 0.73% |
| 2026-02 | 112 | 7,833 | 63 | 49 | 7,770 | 56.2% | 0.80% |
| 2026-03 | 130 | 8,069 | 69 | 61 | 8,000 | 53.1% | 0.86% |
| Q1 distinct HCPs | 291 | 16,723 | 216 | 75 | 16,507 | 74.2% | 1.29% |

* **BlinkRx writers are mostly LAAD writers too:** in a given month about half of BlinkRx writers (44 to 62%) are also LAAD writers of the same type. Over Q1 it is 74% for both paid and non-paid. BlinkRx mostly reaches HCPs already visible in LAAD, not a separate set.
* **BlinkRx adds few new HCPs:** only 45 to 80 paid and 38 to 61 non-paid writers per month appear in BlinkRx alone (91 paid and 75 non-paid over Q1).
* **LAAD is almost untouched by BlinkRx:** BlinkRx covers 0.8 to 1.5% of LAAD Paid Writers and 0.7 to 0.9% of LAAD Non-paid Writers per month (Jaccard 0.7 to 1.5%).
* **Paid overlap is rising:** 44 -> 62 -> 95 shared paid writers from Jan to Mar, in line with BlinkRx growth. Non-paid overlap is flat (62 to 69).
* **Checks:** writer counts on both sides equal the earlier tables (LAAD Steps 2-3, BlinkRx Step B4). A separate set-intersection count gives the same overlap for every month.
* **Caveat:** overlap is at HCP-month level, not patient level (BlinkRx and LAAD patient IDs cannot be linked). An HCP in both may be writing for different patients in each channel.